In [1]:
import pandas as pd
import re
import numpy as np
import torch
import random
from itables import show
import pubchempy as pcp
import rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors
from descriptastorus.descriptors import rdDescriptors
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from functools import lru_cache

In [2]:
# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [3]:
df=pd.read_csv('Cleaned_VLE_Data_with_Smiles_and_Mols.csv')

In [4]:
#To ensure mol are mol objects
df['mol1'] = df['Smiles 1'].apply(Chem.MolFromSmiles)
df['mol2'] = df['Smiles 2'].apply(Chem.MolFromSmiles)

[19:38:53] WARNING: not removing hydrogen atom without neighbors
[19:38:53] WARNING: not removing hydrogen atom without neighbors
[19:38:53] WARNING: not removing hydrogen atom without neighbors
[19:38:53] WARNING: not removing hydrogen atom without neighbors
[19:38:53] WARNING: not removing hydrogen atom without neighbors
[19:38:53] WARNING: not removing hydrogen atom without neighbors
[19:38:53] WARNING: not removing hydrogen atom without neighbors
[19:38:53] WARNING: not removing hydrogen atom without neighbors
[19:38:53] WARNING: not removing hydrogen atom without neighbors
[19:38:53] WARNING: not removing hydrogen atom without neighbors
[19:38:53] WARNING: not removing hydrogen atom without neighbors
[19:38:53] WARNING: not removing hydrogen atom without neighbors
[19:38:53] WARNING: not removing hydrogen atom without neighbors
[19:38:53] WARNING: not removing hydrogen atom without neighbors
[19:38:53] WARNING: not removing hydrogen atom without neighbors
[19:38:53] WARNING: not r

In [5]:
df.head()

,property,value,phase,"Temperature, K","Pressure, kPa",Mole fraction,Component 1,Component 2,Smiles 1,Smiles 2,mol1,mol2
0,"Thermal conductivity, W/m/K",0.02096,Gas,300.0,724.0,0.2493,carbon dioxide,methane,C(=O)=O,C,<rdkit.Chem.rdchem.Mol object at 0x0000026C675...,<rdkit.Chem.rdchem.Mol object at 0x0000026C676...
1,"Thermal conductivity, W/m/K",0.02125,Gas,300.0,1288.0,0.2493,carbon dioxide,methane,C(=O)=O,C,<rdkit.Chem.rdchem.Mol object at 0x0000026C676...,<rdkit.Chem.rdchem.Mol object at 0x0000026C676...
2,"Thermal conductivity, W/m/K",0.02161,Gas,300.0,1835.0,0.2493,carbon dioxide,methane,C(=O)=O,C,<rdkit.Chem.rdchem.Mol object at 0x0000026C676...,<rdkit.Chem.rdchem.Mol object at 0x0000026C676...
3,"Thermal conductivity, W/m/K",0.02197,Gas,300.0,2335.0,0.2493,carbon dioxide,methane,C(=O)=O,C,<rdkit.Chem.rdchem.Mol object at 0x0000026C676...,<rdkit.Chem.rdchem.Mol object at 0x0000026C676...
4,"Thermal conductivity, W/m/K",0.02245,Gas,300.0,2820.0,0.2493,carbon dioxide,methane,C(=O)=O,C,<rdkit.Chem.rdchem.Mol object at 0x0000026C676...,<rdkit.Chem.rdchem.Mol object at 0x0000026C676...


In [6]:
from descriptastorus.descriptors import rdNormalizedDescriptors
generator = rdNormalizedDescriptors.RDKit2DNormalized()
print(generator.columns)

[('BalabanJ', <class 'numpy.float64'>), ('BertzCT', <class 'numpy.float64'>), ('Chi0', <class 'numpy.float64'>), ('Chi0n', <class 'numpy.float64'>), ('Chi0v', <class 'numpy.float64'>), ('Chi1', <class 'numpy.float64'>), ('Chi1n', <class 'numpy.float64'>), ('Chi1v', <class 'numpy.float64'>), ('Chi2n', <class 'numpy.float64'>), ('Chi2v', <class 'numpy.float64'>), ('Chi3n', <class 'numpy.float64'>), ('Chi3v', <class 'numpy.float64'>), ('Chi4n', <class 'numpy.float64'>), ('Chi4v', <class 'numpy.float64'>), ('EState_VSA1', <class 'numpy.float64'>), ('EState_VSA10', <class 'numpy.float64'>), ('EState_VSA11', <class 'numpy.float64'>), ('EState_VSA2', <class 'numpy.float64'>), ('EState_VSA3', <class 'numpy.float64'>), ('EState_VSA4', <class 'numpy.float64'>), ('EState_VSA5', <class 'numpy.float64'>), ('EState_VSA6', <class 'numpy.float64'>), ('EState_VSA7', <class 'numpy.float64'>), ('EState_VSA8', <class 'numpy.float64'>), ('EState_VSA9', <class 'numpy.float64'>), ('ExactMolWt', <class 'numpy

In [7]:
import tqdm.auto as tqdm
# 1. Define a wrapper function and apply the cache to IT
@lru_cache(maxsize=None)
def get_cached_descriptors(mol_input):
    # This function simply calls your generator
    # We hardcode 'None' here since you used it in your original snippet
    return generator.calculateMol(mol_input, None)

# 2. Now run your loops calling the CACHED function
# The first time a specific molecule is seen, it runs the generator.
# The second time (e.g., if the same molecule is in mol2), it returns instantly.

print("Processing Mol 1...")
x_1 = np.stack([get_cached_descriptors(m) for m in tqdm.tqdm(df['mol1'].tolist())])

print("Processing Mol 2...")
x_2 = np.stack([get_cached_descriptors(m) for m in tqdm.tqdm(df['mol2'].tolist())])

# Optional: Check cache stats to see how much time you saved
print(f"\nCache Info: {get_cached_descriptors.cache_info()}")

print(x_1.shape)
print(x_2.shape)

c:\Users\harry\anaconda3\envs\CHE1148\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processing Mol 1...


 66%|██████▌   | 71470/108142 [28:13<11:34, 52.83it/s]  [20:07:22] WARNING: not removing hydrogen atom without neighbors
[20:07:22] WARNING: not removing hydrogen atom without neighbors
[20:07:22] WARNING: not removing hydrogen atom without neighbors
[20:07:22] WARNING: not removing hydrogen atom without neighbors
 66%|██████▌   | 71476/108142 [28:13<11:43, 52.15it/s][20:07:22] WARNING: not removing hydrogen atom without neighbors
[20:07:22] WARNING: not removing hydrogen atom without neighbors
[20:07:22] WARNING: not removing hydrogen atom without neighbors
[20:07:22] WARNING: not removing hydrogen atom without neighbors
[20:07:22] WARNING: not removing hydrogen atom without neighbors
[20:07:22] WARNING: not removing hydrogen atom without neighbors
 66%|██████▌   | 71482/108142 [28:13<11:43, 52.07it/s][20:07:22] WARNING: not removing hydrogen atom without neighbors
[20:07:22] WARNING: not removing hydrogen atom without neighbors
[20:07:22] WARNING: not removing hydrogen atom without n

Processing Mol 2...


100%|██████████| 108142/108142 [35:40<00:00, 50.53it/s] 



Cache Info: CacheInfo(hits=0, misses=216284, maxsize=None, currsize=216284)
(108142, 200)
(108142, 200)


In [8]:
np.save('descriptors_for_comp_1.npy', x_1)
np.save('descriptors_for_comp_2.npy', x_2)

In [9]:
df_mol1 = pd.DataFrame(x_1, columns=generator.columns)
df_mol2 = pd.DataFrame(x_2, columns=generator.columns)
df = pd.concat([df, df_mol1, df_mol2], axis=1)
print(df.shape)
df.head()

(108142, 412)


,property,value,phase,"Temperature, K","Pressure, kPa",Mole fraction,Component 1,Component 2,Smiles 1,Smiles 2,...,"(fr_sulfonamd, <class 'numpy.float64'>)","(fr_sulfone, <class 'numpy.float64'>)","(fr_term_acetylene, <class 'numpy.float64'>)","(fr_tetrazole, <class 'numpy.float64'>)","(fr_thiazole, <class 'numpy.float64'>)","(fr_thiocyan, <class 'numpy.float64'>)","(fr_thiophene, <class 'numpy.float64'>)","(fr_unbrch_alkane, <class 'numpy.float64'>)","(fr_urea, <class 'numpy.float64'>)","(qed, <class 'numpy.float64'>)"
0,"Thermal conductivity, W/m/K",0.02096,Gas,300.0,724.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493
1,"Thermal conductivity, W/m/K",0.02125,Gas,300.0,1288.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493
2,"Thermal conductivity, W/m/K",0.02161,Gas,300.0,1835.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493
3,"Thermal conductivity, W/m/K",0.02197,Gas,300.0,2335.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493
4,"Thermal conductivity, W/m/K",0.02245,Gas,300.0,2820.0,0.2493,carbon dioxide,methane,C(=O)=O,C,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493


In [10]:
df.to_csv('Cleaned_VLE_Data_with_Smiles_and_Mols_and_Descriptors.csv')

In [18]:
df.dropna(inplace=True)
df.shape    
df.to_csv('Cleaned_VLE_Data_with_Smiles_and_Mols_and_Descriptors.csv')

In [19]:
print(df.shape)

(106383, 412)
